In [1]:
import pandas as pd
import os

# read in the sample information: load the csv generated by pypette
df_sample = pd.read_csv("../../../config/samplesTestFastRuns.csv")

In [2]:
df_sample

,sample_name,sample_run,sample_path
0,tinygex,run01,/mnt/SSD01/training/routines/snakemake/sample_...
1,tinygex,run02,/mnt/SSD01/training/routines/snakemake/sample_...
2,tinygex2,run02,/mnt/SSD01/training/routines/snakemake/sample_...
3,tinygex2,run01,/mnt/SSD01/training/routines/snakemake/sample_...
4,tinygex3,run01,/mnt/SSD01/training/routines/snakemake/sample_...
5,tinygex3,run02,/mnt/SSD01/training/routines/snakemake/sample_...
6,tinygex3,run03,/mnt/SSD01/training/routines/snakemake/sample_...
7,tinygex_single,run01,/mnt/SSD01/training/routines/snakemake/sample_...


In [3]:
# --- implementation for single runs ---

# 1. Sort the DataFrame by 'sample_name' and 'sample_run'
df_sample2 = df_sample.sort_values(by=['sample_name', 'sample_run']).reset_index(drop=True)

# 2. Create a cumulative count within each 'sample_name' group: cumcount() starts at 0, so add 1 to start numbering from 1
df_sample2['sample_instance_num'] = df_sample2.groupby('sample_name').cumcount() + 1

# 3. Create the new unique sample name string
df_sample2['sample_name_unique'] = df_sample2['sample_name'] + '_' + df_sample2['sample_instance_num'].astype(str)

# generate a pandas dataframe. Here it is expected only one path per sample/run.
aggregated_single_alt = df_sample2.groupby(['sample_name','sample_run']).agg(
    sample_run=('sample_run', 'unique'),  # Get unique 'sample_run' per group
    sample_path_string=('sample_path', lambda s: sorted(s.map(os.path.dirname).unique())),
    # needed to check for the existance of the actual input files
    sample_path_list=('sample_path','unique'),
    # needed to avoid issues with the *.mro files during processing
    sample_name_unique=('sample_name_unique', 'first')
)

# 3. Convert the aggregated DataFrame to the desired dictionary format
SAMPLES_single = aggregated_single_alt.to_dict('index')

In [4]:
SAMPLES_single

{('tinygex', 'run01'): {'sample_run': array(['run01'], dtype=object),
  'sample_path_string': ['/mnt/SSD01/training/routines/snakemake/sample_data/test_cellranger/test02/run01'],
  'sample_path_list': array(['/mnt/SSD01/training/routines/snakemake/sample_data/test_cellranger/test02/run01/tinygex'],
        dtype=object),
  'sample_name_unique': 'tinygex_1'},
 ('tinygex', 'run02'): {'sample_run': array(['run02'], dtype=object),
  'sample_path_string': ['/mnt/SSD01/training/routines/snakemake/sample_data/test_cellranger/test02/run02'],
  'sample_path_list': array(['/mnt/SSD01/training/routines/snakemake/sample_data/test_cellranger/test02/run02/tinygex'],
        dtype=object),
  'sample_name_unique': 'tinygex_2'},
 ('tinygex2', 'run01'): {'sample_run': array(['run01'], dtype=object),
  'sample_path_string': ['/mnt/SSD01/training/routines/snakemake/sample_data/test_cellranger/test02/run01'],
  'sample_path_list': array(['/mnt/SSD01/training/routines/snakemake/sample_data/test_cellranger/t

In [ ]:
# --- implementation for merging runs ---

# generate a pandas dataframe. Here is expected a comma separated string of paths per sample.
aggregated_merge_alt = df_sample.groupby('sample_name').agg(
    # sample_run=('sample_run', lambda s: ",".join(s.unique())),
    # Aggregate sample_paths:
    # 1. Apply os.path.dirname to each path in the group (s.map(os.path.dirname))
    # 2. Find the unique directory names (.unique())
    # 3. Sort the unique directory names (sorted(...))
    # 4. Join the sorted unique directory names with a comma (",".join(...))
    sample_path_string=('sample_path', lambda s: ",".join(sorted(s.map(os.path.dirname).unique()))),
    # needed to check for the existance of the actual input files
    sample_path_list=('sample_path','unique')
)

# 3. Convert the aggregated DataFrame to the desired dictionary format
SAMPLES_merge = aggregated_merge_alt.to_dict('index')

In [6]:
SAMPLES_merge

{'tinygex': {'sample_path_string': '/mnt/SSD01/training/routines/snakemake/sample_data/test_cellranger/test02/run01,/mnt/SSD01/training/routines/snakemake/sample_data/test_cellranger/test02/run02',
  'sample_path_list': array(['/mnt/SSD01/training/routines/snakemake/sample_data/test_cellranger/test02/run01/tinygex',
         '/mnt/SSD01/training/routines/snakemake/sample_data/test_cellranger/test02/run02/tinygex'],
        dtype=object)},
 'tinygex2': {'sample_path_string': '/mnt/SSD01/training/routines/snakemake/sample_data/test_cellranger/test02/run01,/mnt/SSD01/training/routines/snakemake/sample_data/test_cellranger/test02/run02',
  'sample_path_list': array(['/mnt/SSD01/training/routines/snakemake/sample_data/test_cellranger/test02/run02/tinygex2',
         '/mnt/SSD01/training/routines/snakemake/sample_data/test_cellranger/test02/run01/tinygex2'],
        dtype=object)},
 'tinygex3': {'sample_path_string': '/mnt/SSD01/training/routines/snakemake/sample_data/test_cellranger/test02/